In [ ]:
import pandas as pd

In [ ]:
df_train = pd.read_csv('ttrain_bilingual.csv')
df_val = pd.read_csv('vval_bilingual.csv')
df_test = pd.read_csv('ttest_bilingual.csv')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
class MultilingualDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.texts = df['text'].astype(str).values
        self.labels = df['label'].astype(int).values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
class XLMRobertaClassifier(nn.Module):
    def __init__(self, model_name='xlm_roberta-base', num_classes=3, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        x = self.dropout(mean_pooled)
        return self.fc(x)

In [ ]:
def train(model, data_loader, loss_fn, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct_preds = 0
    total_samples = 0

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        _, preds = torch.max(outputs, dim=1)
        total_loss += loss.item()
        correct_preds += torch.sum(preds == labels).item()
        total_samples += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct_preds / total_samples
    return avg_loss, accuracy

In [ ]:
def evaluate(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs, labels)

            _, preds = torch.max(outputs, dim=1)

            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

    return avg_loss, acc, f1, precision, recall

In [ ]:
model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

batch_size = 16
epochs = 4

train_dataset = MultilingualDataset(df_train, tokenizer, max_len=512)
val_dataset = MultilingualDataset(df_val, tokenizer, max_len=512)
test_dataset = MultilingualDataset(df_test, tokenizer, max_len=512)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [ ]:
model = XLMRobertaClassifier(
    model_name=model_name,
    num_classes=3,
    dropout=0.2
).to(device)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_train['label']),
    y=df_train['label']
)

weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

loss_fn = nn.CrossEntropyLoss(weight=weights_tensor)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.5e-5,
    weight_decay=0.02
)

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

best_val_f1 = 0.0

for epoch in range(epochs):

    print(f"\nEpoch {epoch + 1}/{epochs}")

    train_loss, train_acc = train(
        model,
        train_loader,
        loss_fn,
        optimizer,
        scheduler=scheduler,
        device=device
    )

    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(
        model,
        val_loader,
        loss_fn,
        device
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Val Precision: {val_prec:.4f} | "
        f"Val Recall: {val_rec:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print(
            f"Best model saved! "
            f"Validation F1: {val_f1:.4f}"
        )


model.load_state_dict(
    torch.load("best_model.pth")
)

test_loss, test_acc, test_f1, test_prec, test_rec = evaluate(
    model,
    test_loader,
    loss_fn,
    device
)

print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Test F1        : {test_f1:.4f}")
print(f"Test Precision  : {test_prec:.4f}")
print(f"Test Recall     : {test_rec:.4f}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/4


Evaluating: 100%|██████████| 19/19 [00:08<00:00,  2.18it/s]


... (5 outputs left out)

Evaluating: 100%|██████████| 19/19 [00:08<00:00,  2.19it/s]


Train Loss: 0.0629 | Train Acc: 0.9817
Val Loss: 0.0700 | Val Acc: 0.9833 | Val F1: 0.9833 | Val Precision: 0.9834 | Val Recall: 0.9833
Best model saved! Validation F1: 0.9833


Evaluating: 100%|██████████| 19/19 [00:08<00:00,  2.26it/s]


========== FINAL TEST RESULTS ==========
Test Accuracy  : 0.9833
Test F1        : 0.9833
Test Precision  : 0.9834
Test Recall     : 0.9833


In [ ]:
torch.save(model, "best_model.pth")